# Fáze 9: Porovnání plného a cold-start modelu

10 produktů (2 per třída) klasifikovaných oběma modely.

In [ ]:
import os, sys
from pathlib import Path
_PROJECT_ROOT = Path(os.path.abspath("")).parent if Path(os.path.abspath("")).name == "notebooks" else Path(os.path.abspath(""))
os.chdir(_PROJECT_ROOT)
sys.path.insert(0, str(_PROJECT_ROOT))

import pandas as pd
import numpy as np

CLASS_CZ = {
    "shelf_picking": "Police",
    "front_zone_bin": "Prihradka",
    "special_zone": "Specialni zona",
    "floor_block": "Blokova zona",
    "pallet_rack": "Paleta",
}

## 1. Plný model (s obrátkovostí, F1=0.9866)

In [ ]:
df = pd.read_csv("results/phase9_prototype/validation_table.csv")
print(f"Presnost: {df['correct'].mean():.0%} ({df['correct'].sum()}/{len(df)})")

table = df[["product_idx", "true_class", "predicted_class", "confidence", "knn_agreement", "correct"]].copy()
table["true_class"] = table["true_class"].map(CLASS_CZ)
table["predicted_class"] = table["predicted_class"].map(CLASS_CZ)
table["confidence"] = (table["confidence"] * 100).round(1).astype(str) + " %"
table["knn_agreement"] = (table["knn_agreement"] * 100).astype(int).astype(str) + " %"
table["correct"] = table["correct"].map({True: "Ano", False: "Ne"})
table.columns = ["ID", "Skutecna trida", "Predikce", "Confidence", "KNN shoda", "Spravne"]
table.style.set_caption("Plny model (s obratkovosti)")

## 2. Cold-start model (bez obrátkovosti, F1=0.8305)

Stejných 10 produktů, ale klasifikováno modelem natrénovaným BEZ daily_turnover.

In [ ]:
df_cs = pd.read_csv("results/phase9_prototype/validation_table_cold_start.csv")
print(f"Presnost: {df_cs['correct'].mean():.0%} ({df_cs['correct'].sum()}/{len(df_cs)})")

table_cs = df_cs[["product_idx", "true_class", "predicted_class", "confidence", "knn_agreement", "correct"]].copy()
table_cs["true_class"] = table_cs["true_class"].map(CLASS_CZ)
table_cs["predicted_class"] = table_cs["predicted_class"].map(CLASS_CZ)
table_cs["confidence"] = (table_cs["confidence"] * 100).round(1).astype(str) + " %"
table_cs["knn_agreement"] = (table_cs["knn_agreement"] * 100).astype(int).astype(str) + " %"
table_cs["correct"] = table_cs["correct"].map({True: "Ano", False: "Ne"})
table_cs.columns = ["ID", "Skutecna trida", "Predikce", "Confidence", "KNN shoda", "Spravne"]
table_cs.style.set_caption("Cold-start model (bez obratkovosti)")

## 3. Porovnání: plný vs. cold-start model

In [ ]:
compare = pd.DataFrame({
    "ID": df["product_idx"],
    "Skutecna trida": df["true_class"].map(CLASS_CZ),
    "Plny model": df["predicted_class"].map(CLASS_CZ),
    "Plny conf.": (df["confidence"] * 100).round(1).astype(str) + " %",
    "Cold-start": df_cs["predicted_class"].map(CLASS_CZ),
    "CS conf.": (df_cs["confidence"] * 100).round(1).astype(str) + " %",
    "Oba spravne": (df["correct"] & df_cs["correct"]).map({True: "Ano", False: "Ne"}),
})
compare.style.set_caption("Porovnani plneho a cold-start modelu na 10 produktech")